# PPSimExample

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `smoke`
- Workflow family: `network`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/PPSimExample.md`


Notebook source link: [PPSimExample.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/PPSimExample.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "PPSimExample"
FAMILY = "network"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"PPSimExample: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"PPSimExample: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"PPSimExample: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"PPSimExample: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "Ts=.001;            %Sample Time",
    "tMin=0; tMax=50;    %Simulation duration",
    "t=tMin:Ts:tMax;",
    "mu=-3;              %Baseline firing rate of the neurons being modeled",
    "H=tf([-1 -2 -4],[1],Ts,'Variable','z^-1');",
    "S=tf([1],1,Ts,'Variable','z^-1');",
    "E=tf([0],1,Ts,'Variable','z^-1');",
    "f=1;                    %Stimulus frequency",
    "u = sin(2*pi*f*t)';       %Make this neuron modulated by a sine wave",
    "e = zeros(length(t),1);   %No Ensemble input",
    "stim=Covariate(t',u,'Stimulus','time','s','Voltage',{'sin'});",
    "ens =Covariate(t',e,'Ensemble','time','s','Spikes',{'n1'});",
    "numRealizations = 5;    %Number of sample paths to generate",
    "fitType = 'binomial';",
    "sC=CIF.simulateCIF(mu,H,S,E,stim,ens,numRealizations,fitType);",
    "figure;",
    "subplot(2,1,1); sC.plot;    v=axis; axis([0 tMax/10 v(3) v(4)]);",
    "subplot(2,1,2); stim.plot;  v=axis; axis([0 tMax/10 v(3) v(4)]);",
    "baseline=Covariate(t',ones(length(t),1),'Baseline','time','s','',{'mu'});",
    "spikeColl = sC;               %Use the generated data as our collection of spikes",
    "cc=CovColl({stim,baseline});  %Use stimulation and baseline as possible covariates",
    "trial = Trial(spikeColl,cc); sampleRate = 1/Ts; %Create trial",
    "clear c;",
    "selfHist = [0:0.001:0.003]; %We know the history effect goes back 3 lag orders",
    "c{1} = TrialConfig({{'Baseline','mu'}},sampleRate,[],[]);",
    "c{1}.setName('Baseline');",
    "c{2} = TrialConfig({{'Baseline','mu'},{'Stimulus','sin'}},sampleRate,[],[]);",
    "c{2}.setName('Stim');",
    "c{3} = TrialConfig({{'Baseline','mu'},{'Stimulus','sin'}},sampleRate,selfHist,[]);",
    "c{3}.setName('Stim+Hist');",
    "cfgColl= ConfigColl(c);",
    "if(strcmp(fitType,'binomial'))",
    "Algorithm = 'BNLRCG';   % BNLRCG - faster Truncated, L-2 Regularized,",
    "else",
    "Algorithm = 'GLM';      % Standard Matlab GLM (Can be used for binomial or",
    "end",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0,Algorithm);",
    "results{1}.plotResults;",
    "Summary = FitResSummary(results);",
    "Summary.plotSummary;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for PPSimExample.")


In [ ]:
# PPSimExample: fixture-backed Poisson GLM simulation and parity checks.
from pathlib import Path
import nstat
from scipy.io import loadmat
fixture_path = Path(nstat.__file__).resolve().parents[2] / "tests/parity/fixtures/matlab_gold/PPSimExample_gold.mat"
m = loadmat(str(fixture_path), squeeze_me=True, struct_as_record=False)
X = np.asarray(m["X"], dtype=float).reshape(-1, 1)
y = np.asarray(m["y"], dtype=float).reshape(-1)
dt = float(np.asarray(m["dt"], dtype=float).reshape(-1)[0])
expected_rate = np.asarray(m["expected_rate"], dtype=float).reshape(-1)
b = np.asarray(m["b"], dtype=float).reshape(-1)
fit = Analysis.fit_glm(X=X, y=y, fit_type="poisson", dt=dt)
pred_rate = np.asarray(fit.predict(X), dtype=float).reshape(-1)
rel_err = float(np.mean(np.abs(pred_rate - expected_rate) / np.maximum(expected_rate, 1e-12)))
intercept_abs_error = float(abs(fit.intercept - b[0]))
coeff_abs_error = float(abs(fit.coefficients[0] - b[1]))
assert rel_err <= 0.25 and intercept_abs_error <= 0.25 and coeff_abs_error <= 0.25
time = np.arange(X.shape[0]) * dt
stim = X.reshape(-1)
spike_idx = np.where(y > 0)[0]

fig, axes = plt.subplots(3, 1, figsize=(10.2, 7.4), sharex=False)
axes[0].plot(time, stim, "k", linewidth=1.0)
axes[0].set_title(f"{TOPIC}: driving stimulus")
axes[0].set_ylabel("stim")
axes[1].vlines(time[spike_idx], 0.6, 1.4, color="black", linewidth=0.35)
axes[1].set_title("Point-process sample path")
axes[1].set_ylabel("trial #1")
axes[2].plot(time, expected_rate, color="tab:green", linewidth=1.0, linestyle="--", label="MATLAB gold")
axes[2].plot(time, pred_rate, color="tab:red", linewidth=1.0, label="Python fit")
axes[2].plot(time, y / max(dt, 1e-12), color="0.7", linewidth=0.3, alpha=0.5, label="counts/dt")
axes[2].set_xlabel("time [s]")
axes[2].set_ylabel("Hz")
axes[2].set_title("Conditional intensity fit")
axes[2].legend(loc="upper right")
plt.tight_layout()
plt.show()

CHECKPOINT_METRICS = {
    "mean_simulated_rate": float(np.mean(pred_rate)),
    "relative_rate_error": rel_err,
    "intercept_abs_error": intercept_abs_error,
    "coeff_abs_error": coeff_abs_error,
}
CHECKPOINT_LIMITS = {
    "mean_simulated_rate": (0.1, 500.0),
    "relative_rate_error": (0.0, 0.25),
    "intercept_abs_error": (0.0, 0.25),
    "coeff_abs_error": (0.0, 0.25),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
